# edge3/edge4 — 10年検定ランナー（E3A–E3D）

事前登録: `docs/26_preregistration_edge3_edge4.md`（**LOCKED**, N=4, α=0.0125）。

**規律（厳守）**
- 拘束力ある判定は**この10年実データ実行**で確定。短期の好成績は楽観側の上限としてのみ扱う。
- 数値は各 runner が出す **JSON を単一スカラで直読**して転記（表示破損のまま転記しない）。
- 当たるまで候補を増やさない／ルールを後出しで変えない。全REJECTなら「3本目なし」と正直に結論。

実行順: ①Drive マウント → ②設定 → ③依存導入 → ④4本実行 → ⑤サマリ。

## ① Google Drive をマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## ② パス設定（ここだけ環境に合わせて編集）

- `REPO_DIR`  : このリポジトリ（research/ がある場所）。GitHub から clone するか、Drive 上のパスを指定。
- `DATA_DIR`  : `research/data/README.md` の通り 10年CSV を置いた場所。
- `REPORTS_DIR`: 結果 JSON の出力先（Drive 推奨＝消えない）。

In [ ]:
import os, sys, subprocess

# --- 編集する3行 ---
REPO_DIR    = '/content/chien-monitor'                                   # repo の場所
DATA_DIR    = '/content/drive/MyDrive/edge_research/data'                # 10年データ
REPORTS_DIR = '/content/drive/MyDrive/edge_research/reports'            # 出力先
GIT_URL     = 'https://github.com/iq87jun-star/chien-monitor.git'       # clone 元（任意）
GIT_BRANCH  = 'claude/new-session-7rAJ3'
# --------------------

# REPO_DIR が無ければ clone（private の場合は事前に PAT を設定 or 手動 upload）
if not os.path.isdir(os.path.join(REPO_DIR, 'research')):
    print('cloning', GIT_URL)
    subprocess.run(['git', 'clone', '--branch', GIT_BRANCH, GIT_URL, REPO_DIR], check=True)

os.makedirs(REPORTS_DIR, exist_ok=True)
assert os.path.isdir(os.path.join(REPO_DIR, 'research')), f'research/ が見つからない: {REPO_DIR}'
assert os.path.isdir(DATA_DIR), f'DATA_DIR が見つからない: {DATA_DIR}'
print('OK  repo =', REPO_DIR, '\n    data =', DATA_DIR, '\n    reports =', REPORTS_DIR)
print('data files:', sorted(os.listdir(DATA_DIR))[:30])

## ③ 依存導入

In [ ]:
!pip -q install numpy pandas scipy

## ④ 4本を実行（E3A → E3B → E3C → E3D）

env を import 前に設定する必要があるため、`data_io` と各 runner を毎回 reload する。

In [ ]:
import importlib, json, traceback

os.environ['EDGE_DATA_DIR'] = DATA_DIR
os.environ['EDGE_REPORTS_DIR'] = REPORTS_DIR
research_dir = os.path.join(REPO_DIR, 'research')
if research_dir not in sys.path:
    sys.path.insert(0, research_dir)

RUNNERS = [
    ('edge3a_eqmr_10y',    'edge3a_result.json', 'E3A 株価指数MR'),
    ('edge3b_leadlag_10y', 'edge3b_result.json', 'E3B リードラグ'),
    ('edge3c_carry_10y',   'edge3c_result.json', 'E3C 横断キャリー(spot)'),
    ('edge3d_session_10y', 'edge3d_result.json', 'E3D セッションBO'),
]

results = {}
for mod_name, out_json, label in RUNNERS:
    print(f'\n===== {label} ({mod_name}) =====')
    try:
        import data_io; importlib.reload(data_io)
        mod = importlib.import_module(mod_name); importlib.reload(mod)
        mod.main()
        with open(os.path.join(REPORTS_DIR, out_json)) as f:
            results[mod_name] = json.load(f)
    except Exception as e:
        print('FAILED:', e); traceback.print_exc()
        results[mod_name] = {'error': str(e)}
print('\nDONE. JSON は', REPORTS_DIR, 'に保存済み。')

## ⑤ サマリ（このセル出力をそのまま貼ってください）

ADOPT 判定は全6ゲート通過のみ。LEAD = 純益>0 かつ p<0.10。それ以外 REJECT。

In [ ]:
import pandas as pd
rows = []
for mod_name, out_json, label in RUNNERS:
    r = results.get(mod_name, {})
    if 'error' in r:
        rows.append({'edge': label, 'grade': 'ERROR', 'note': r['error'][:60]}); continue
    g = r.get('gates', {})
    rows.append({
        'edge': label,
        'grade': r.get('grade'),
        'n': r.get('n_trades'),
        'net_10y': round(r.get('net_profit_10y', float('nan')), 4),
        'p_net': round(r.get('perm_p_net', float('nan')), 4),
        'jk_maxp': round(r.get('jackknife_max_p', float('nan')), 4),
        'corr_v7': round(r.get('corr_v7', float('nan')), 3),
        'gates_pass': sum(bool(v) for v in g.values()),
        'g1g2g3g4g5g6': ''.join('1' if g.get(k) else '0' for k in
            ['g1_profit','g2_perm_p','g3_jackknife','g4_placebo','g5_cost','g6_independent']),
    })
summary = pd.DataFrame(rows)
print('alpha = 0.0125 (Bonferroni 0.05/4) | period = 2016-01..2025-12')
print(summary.to_string(index=False))
print('\n--- 生 JSON（転記用・単一スカラ）---')
print(json.dumps(results, ensure_ascii=False, indent=2))